# Room Booking Chatbot — Technology Walkthrough

This notebook explains the key technologies used in the solution and shows them in action with runnable code examples.

**Stack:** Python · LangChain · LangGraph · Groq (Llama 3.3) · SQLite · Streamlit

---
## 1. LLM Provider — Groq

Groq is a free LLM inference API (no credit card required). It supports OpenAI-compatible requests and integrates natively with LangChain via `langchain-groq`.

We use the `llama-3.3-70b-versatile` model, which has strong tool-calling support — meaning it can reliably choose which function to invoke and with what arguments based on natural language input.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0,
)

response = llm.invoke("What is the capital of France?")
print(response.content)

---
## 2. Tool Calling with LangChain

Tool calling (also called function calling) lets the LLM decide to invoke a predefined Python function instead of generating a text response. The LLM receives the tool's name and docstring, and when it decides to use it, it outputs a structured JSON payload with the arguments.

LangChain's `@tool` decorator turns any Python function into a tool the LLM can call.

In [ ]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city.

    Args:
        city: The name of the city.
    """
    # In a real app this would call a weather API
    return f"It is sunny and 22°C in {city}."

# The LLM sees this schema when deciding whether to call the tool:
print(get_weather.name)
print(get_weather.description)

---
## 3. The ReAct Agent Pattern

The solution uses a **ReAct** (Reason + Act) agent via `langgraph.prebuilt.create_react_agent`. Each turn the agent:
1. **Thinks** about what to do next
2. **Calls a tool** (Action)
3. **Observes** the result
4. Repeats until it has enough information to give a **Final Answer**

This loop allows the agent to chain multiple tool calls in one conversational turn — for example, checking available rooms before creating a booking.

LangGraph's `create_react_agent` is the modern replacement for the deprecated `AgentExecutor`. It returns a compiled graph that accepts a list of messages and returns the full updated message list.

In [ ]:
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

# create_react_agent compiles a LangGraph StateGraph that runs the
# Reason → Act → Observe loop automatically.
tools = [get_weather]
graph = create_react_agent(llm, tools, prompt="You are a helpful assistant.")

# The graph exposes its nodes — 'agent' (LLM call) and 'tools' (tool execution)
print("Graph nodes:", list(graph.nodes.keys()))
print("Bound tools:", [t.name for t in tools])

# When invoked, the graph accepts a list of messages and returns the updated list.
# The full live demo with our actual booking tools is shown in Section 6.

---
## 4. Booking Store — SQLite Layer

All booking logic lives in `booking_store.py` and uses SQLite for persistence. The store enforces every business rule deterministically — the LLM cannot create an invalid booking because the tools return structured errors if constraints are violated.

In [ ]:
import tempfile
from src import booking_store

# Use a temp file so data persists across all calls in this notebook session.
# (SQLite ':memory:' creates a new empty DB per connection, so it cannot be shared.)
_tmp_db = tempfile.mktemp(suffix=".db")
booking_store.DB_PATH = _tmp_db
booking_store.init_db()

print("Rooms and capacities:", booking_store.ROOMS)

In [ ]:
# Create a valid booking
result = booking_store.create_booking(
    room="B",
    title="Team Sync",
    attendees=4,
    start_dt="2026-07-20 10:00",
    end_dt="2026-07-20 11:00",
    owner="User1",
)
print("Created:", result)

In [ ]:
# Try to double-book the same room — should fail
result = booking_store.create_booking(
    room="B",
    title="Another Meeting",
    attendees=2,
    start_dt="2026-07-20 10:30",
    end_dt="2026-07-20 11:30",
    owner="User2",
)
print("Overlap attempt:", result)

In [ ]:
# List available rooms for a time range
available = booking_store.list_available_rooms(
    start_dt="2026-07-20 10:00",
    end_dt="2026-07-20 11:00",
    attendees=4,
)
print("Available rooms (10:00–11:00, 4 people):", available)

In [ ]:
# View room B's schedule for the day
schedule = booking_store.get_room_schedule("B", "2026-07-20")
# Print just the occupied slots for brevity
occupied = [s for s in schedule["slots"] if s["status"] == "occupied"]
print(f"Occupied slots in Room B on 2026-07-20: {occupied}")

In [ ]:
# List User1's bookings
my_bookings = booking_store.list_my_bookings("User1")
print("User1 bookings:", my_bookings)

---
## 5. Tool Binding per User

Each Streamlit session creates a fresh set of tools via `make_tools(current_user)`. The `current_user` is captured in a closure, so the LLM never sees or controls which user is performing the action — preventing identity spoofing.

In [ ]:
from src.tools import make_tools

user1_tools = make_tools("User1")
print("Tools available to User1:")
for t in user1_tools:
    print(f"  {t.name}: {t.description[:60]}...")

---
## 6. End-to-End Agent Interaction

Here is what a real conversation looks like when the agent is wired up with Groq and the booking tools. (Requires a valid `GROQ_API_KEY` in `.env`.)

In [ ]:
from src.agent import build_agent, run_agent

graph = build_agent("User1")
history = []

questions = [
    "What rooms are available on 2026-07-20 at 2pm for 5 people for 1 hour?",
    "Book Room C for a Product Review meeting with 5 attendees from 14:00 to 15:00 on 2026-07-20.",
    "Show me my bookings.",
]

for q in questions:
    print(f"\nUser: {q}")
    answer = run_agent(graph, q, history, "User1")
    print(f"Assistant: {answer}")
    history.append({"role": "user", "content": q})
    history.append({"role": "assistant", "content": answer})